# 7.3.2 — Ensemble Methods

**Learning goals**
- Explain why combining several models can improve predictions.
- Distinguish **bagging** (independent models) from **boosting** (sequential models).
- Train and evaluate a Random Forest, then practise AdaBoost and Gradient Boosting.
- Compare models using the **same features, train/test split and metrics**.

**Connection with 7.3.1:** a single decision tree can underfit when it is too shallow and overfit when it is too deep. What if we combine many trees instead?

## 1. One tree or many?

A **decision tree** learns a sequence of rules. It is easy to visualize, but a small change in the training data can lead to different splits.

An **ensemble** combines predictions from multiple estimators:

- **Regression:** often average their predictions (bagging), or add successive corrections (boosting).
- **Classification:** often combine votes or class probabilities; the precise rule depends on the algorithm.

**Two families:**

| Bagging / Random Forest | Boosting |
|---|---|
| Fit many models on resampled data (typically independently) | Fit models sequentially |
| Reduce instability / variance | Improve the combined prediction by correcting previous weaknesses |
| Combine predictions by averaging or voting | Combine a sequence of weighted or additive predictions |

We will continue with **California Housing regression** to compare with 7.3.1. The lab later uses **classification**; the same ensemble families have classifier counterparts.

## 2. Prepare the familiar data — once

This notebook runs **independently** of 7.3.1. We use exactly the same five illustrative features, split and random seed as in that notebook.

Trees do **not** require feature scaling: their decisions depend on thresholds and ordering, not distances. We therefore use a `Pipeline` containing only the estimator. Different algorithms can use different preprocessing pipelines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, BaggingRegressor,
    AdaBoostRegressor, GradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE = 42
N_ESTIMATORS = 60  # Fixed for today's demonstration; tuning comes tomorrow.

housing = fetch_california_housing(as_frame=True)
X = housing.data[["MedInc", "HouseAge", "AveRooms", "Latitude", "Longitude"]]
y = housing.target  # Median house value, in units of $100,000.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

### One shared evaluation function

To compare models fairly, use the **same test set** and **the same metrics**. MAE is the mean absolute error (lower is better); R² measures predictive fit (higher is better).

For today's lesson, we evaluate the fixed configurations on the test set **once**. On Thursday, we will use cross-validation on the training data to select hyperparameters, reserving the test set for final evaluation.

In [ ]:
results = []

def evaluate_regressor(name, fitted_pipeline):
    """Record train and test MAE, plus test R², for a fitted pipeline."""
    pred_train = fitted_pipeline.predict(X_train)
    pred_test = fitted_pipeline.predict(X_test)
    scores = {
        "Model": name,
        "Train MAE": mean_absolute_error(y_train, pred_train),
        "Test MAE": mean_absolute_error(y_test, pred_test),
        "Test R2": r2_score(y_test, pred_test),
    }
    results.append(scores)
    return pd.Series(scores)

# Baseline: the same small tree used for illustration in 7.3.1.
single_tree_pipeline = Pipeline([
    ("model", DecisionTreeRegressor(max_depth=3, random_state=RANDOM_STATE)),
])
single_tree_pipeline.fit(X_train, y_train)
evaluate_regressor("Decision Tree (depth=3)", single_tree_pipeline)

## 3. Random Forest: many different trees

A **Random Forest** trains many decision trees. Each tree typically sees a bootstrap sample of observations, and only a random subset of features is considered at each split. This helps the trees make **different errors**; averaging their predictions often produces a more stable model.

We fix `n_estimators` (number of trees) and `max_depth` (maximum depth of each tree). These are **illustrative settings, not optimized values**.

In [ ]:
forest_pipeline = Pipeline([
    ("model", RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=6,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])
forest_pipeline.fit(X_train, y_train)
evaluate_regressor("Random Forest", forest_pipeline)

### What does the forest consider important?

Like a decision tree, a Random Forest provides `feature_importances_`. These values measure the **total impurity reduction attributed to each feature** across its trees. They are nonnegative and sum to 1.

They are **not causal effects** and can be misleading when features are correlated or have different numbers of possible split points.

In [ ]:
forest = forest_pipeline.named_steps["model"]
forest_importances = pd.Series(
    forest.feature_importances_, index=X_train.columns
).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
forest_importances.plot.barh(ax=ax, color="steelblue")
ax.set(xlabel="Impurity-based feature importance",
       title="Random Forest — feature importance")
plt.tight_layout()
plt.show()

**Check for understanding (1 minute)**

1. Why might averaging many trees generalize better than using a single deep tree?
2. Does a Random Forest need `StandardScaler`? Why or why not?

## 4. Bagging and Pasting

**Bagging** (*bootstrap aggregating*) trains several estimators on different samples of the training set **with replacement**. **Pasting** samples **without replacement**. The predictions are then aggregated.

A Random Forest is a specialized ensemble of randomized decision trees. Bagging is more general: the base estimator does not have to be a tree.

The classification lab asks you to try bagging/pasting. Here is the minimal syntax, with fixed parameters; we will not spend time tuning it today.

In [ ]:
# In recent scikit-learn versions, use estimator= (not base_estimator=).
bagging_pipeline = Pipeline([
    ("model", BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
        n_estimators=N_ESTIMATORS,
        bootstrap=True,  # True = bagging; False = pasting.
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])
bagging_pipeline.fit(X_train, y_train)
evaluate_regressor("Bagging", bagging_pipeline)

**Bonus / self-learning:** Compare `bootstrap=True` and `bootstrap=False` and inspect the official documentation:

- [BaggingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingRegressor.html)
- [BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)

For classification, replace the regressor with `BaggingClassifier` and use a suitable classification metric.

## 5. Boosting: learning in sequence

Unlike bagging, boosting trains estimators **sequentially**. Each new estimator is influenced by the performance of the ensemble built so far.

Two important approaches:

- **AdaBoost:** increases attention to observations that were difficult for earlier estimators (in regression, the algorithm adjusts sample weights using prediction errors).
- **Gradient Boosting:** adds new estimators that move the combined prediction in the direction that reduces a chosen loss function. For squared-error regression, this can be understood as fitting the residuals.

We will use **small trees** and fixed settings. Hyperparameter optimization is tomorrow's topic.

### A visual intuition: residual correction

Suppose the first model predicts `2.0` for a home whose actual target is `2.8` (in units of $100,000). Its residual is `+0.8`. A subsequent boosting stage can learn a correction in that direction.

This is a **conceptual illustration**: the precise correction depends on the loss function and the learning rate, not simply on adding the full residual.

## 6. Guided practice — AdaBoost and Gradient Boosting (10–15 minutes)

**Your task:** complete the two pipelines below, train them on `X_train`, and compare their results with the previous models.

Use these **fixed values**:

- `random_state=RANDOM_STATE`
- `n_estimators=N_ESTIMATORS` (60)
- AdaBoost base tree: `max_depth=2`
- Gradient Boosting trees: `max_depth=2`, `learning_rate=0.1`

Do **not** search for better parameters yet. That is the purpose of Thursday's lesson.

Documentation: [AdaBoostRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostRegressor.html) · [GradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)

In [ ]:
# Guided example A — AdaBoost
# Fill in the TODOs, then run this cell.
ada_pipeline = Pipeline([
    ("model", AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=2, random_state=RANDOM_STATE),
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
    )),
])

# TODO: Fit the pipeline on the training data.
# ada_pipeline.fit(...)

# TODO: Uncomment after fitting to record its scores.
# evaluate_regressor("AdaBoost", ada_pipeline)

In [ ]:
# Guided example B — Gradient Boosting
# GradientBoostingRegressor creates its own regression trees internally.
gradient_pipeline = Pipeline([
    ("model", GradientBoostingRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=2,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
    )),
])

# TODO: Fit the pipeline on the training data.
# gradient_pipeline.fit(...)

# TODO: Uncomment after fitting to record its scores.
# evaluate_regressor("Gradient Boosting", gradient_pipeline)

### Compare your results

Once you have fitted the two boosting models, the following table will include them automatically. Compare **test MAE**, and look at the difference between **train MAE** and **test MAE**.

**Questions (2 minutes)**
1. Which model has the lowest test MAE in this comparison?
2. Does the model with the smallest training error also have the smallest test error?
3. Why would we need cross-validation before claiming that one configuration is the best?

In [ ]:
comparison = pd.DataFrame(results).set_index("Model")
comparison.sort_values("Test MAE").round(3)

## 7. From regression examples to the classification lab

The lab uses **Spaceship Titanic**, a **classification** problem. The ensemble concepts are the same, but you will use the classifier variants:

| Today's regression example | Classification counterpart |
|---|---|
| `BaggingRegressor` | [`BaggingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html) |
| `RandomForestRegressor` | [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) |
| `AdaBoostRegressor` | [`AdaBoostClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) |
| `GradientBoostingRegressor` | [`GradientBoostingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) |

**Remember:** split first; fit learned preprocessing steps on **training data only**, preferably inside a pipeline. Choose **classification metrics** (such as precision, recall and F1) rather than MAE/R². The appropriate metric depends on the problem.

A tree-based model usually does not need numerical scaling, but your pipeline may still need **imputation or categorical encoding**, depending on the dataset.

## 8. Wrap-up

- **One decision tree:** simple to interpret, but can be unstable or overfit.
- **Bagging / Random Forest:** combine many varied estimators, often reducing variance.
- **Boosting:** builds an ensemble sequentially, focusing on improving the combined predictions.
- **Fair comparisons:** compare the full combination of **features + preprocessing + model**, on consistent data splits and metrics.

**Next: Hyperparameter Tuning and Cross-Validation.** Today we fixed model settings. Tomorrow we will learn to select them using training data and cross-validation rather than repeatedly adjusting them against the test set.

**Optional exploration:** [scikit-learn Ensemble User Guide](https://scikit-learn.org/stable/modules/ensemble.html).